# BC Training — Kaggle

Запускает supervised обучение политики на собранном датасете.

**Перед запуском:**
1. Загрузи на Kaggle Datasets три файла: `bc_dataset.npz`, `bc_dataset_obs.dat`, `bc_dataset_act.dat`
2. Добавь датасет к этому notebook через Add Data
3. Включи Accelerator: P100
4. Включи Internet: On (Settings → Internet)

In [ ]:
# Проверяем GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не найден — включи P100 в настройках!'}")

In [ ]:
# Клонируем репо и устанавливаем зависимости
!git clone https://github.com/Andrew82mm/RL_practice.git /kaggle/working/RL_practice
%cd /kaggle/working/RL_practice
!pip install -q sb3-contrib gymnasium torch scipy tqdm

In [ ]:
import os, shutil, numpy as np

# Имя датасета на Kaggle: /kaggle/input/<dataset-slug>/
# Замени 'bc-dataset-blockpuzzle' на реальное slug-имя своего датасета
KAGGLE_DATASET_DIR = "/kaggle/input/bc-dataset-blockpuzzle"
LOCAL_DIR = "/kaggle/working/dataset"
os.makedirs(LOCAL_DIR, exist_ok=True)

# Копируем на локальный диск (важно для скорости случайных чтений DataLoader)
print("Копируем данные (~7.2 ГБ)...")
for fname in ["bc_dataset.npz", "bc_dataset_obs.dat", "bc_dataset_act.dat"]:
    dst = f"{LOCAL_DIR}/{fname}"
    if not os.path.exists(dst):
        shutil.copy2(f"{KAGGLE_DATASET_DIR}/{fname}", dst)
    print(f"  {fname}: {os.path.getsize(dst)/1024**2:.0f} MB — OK")

# Патчим пути в .npz
meta = np.load(f"{LOCAL_DIR}/bc_dataset.npz", allow_pickle=True)
np.savez(f"{LOCAL_DIR}/bc_dataset.npz",
    n_samples=meta["n_samples"],
    obs_shape=meta["obs_shape"],
    obs_path=np.array(f"{LOCAL_DIR}/bc_dataset_obs.dat"),
    act_path=np.array(f"{LOCAL_DIR}/bc_dataset_act.dat"),
)
print(f"\nДатасет готов: {int(meta['n_samples']):,} пар")

In [ ]:
# Запускаем BC обучение
# Модель сохраняется в /kaggle/working/ — скачай через Output после завершения
!python training/bc_train.py \
    --only-train \
    --arch vit \
    --dataset-path "/kaggle/working/dataset/bc_dataset.npz" \
    --pretrained-path "/kaggle/working/vit_bc_pretrained"

In [ ]:
# Проверяем результат
import os
model_path = "/kaggle/working/vit_bc_pretrained.zip"
print(f"Модель: {os.path.getsize(model_path)/1024**2:.1f} MB")
print("Скачай vit_bc_pretrained.zip через вкладку Output →")